# Ranking: Feature Engineering

In [1]:
import sys
import os
%load_ext autoreload
%autoreload 2

sys.path.append(os.path.abspath("../src"))

## Spark + Data Initialisation
Initialising Spark and loading data saved as parquets from retrieval stage

In [2]:
from utils.spark_session import get_spark

spark = get_spark()
spark.sparkContext.setLogLevel("INFO")

In [3]:
# Reloading data from ALS retrieval stage:
# recommendations per user from ALS
# test data

als_candidates = spark.read.parquet('../data/retrieval/als_candidates.parquet')
test = spark.read.parquet('../data/retrieval/test_filtered.parquet')
train = spark.read.parquet('../data/retrieval/train.parquet')

## User Features

In [4]:
# Dataframe for quick development
import configs.settings as cfg
user_sample = train.select(cfg.USER_COL).distinct().sample(0.01, seed=10)

train_sample = train.join(user_sample, on=cfg.USER_COL, how='inner')
train_sample.show(5)

+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|   125|   3489|   4.5|2010-07-14 16:01:48|
|   125|    230|   2.5|2010-07-14 16:01:59|
|   125|   2160|   1.0|2010-07-14 16:02:11|
|   125|   3247|   3.0|2010-07-14 16:02:22|
|   125|   1590|   2.0|2010-07-14 16:02:44|
+------+-------+------+-------------------+
only showing top 5 rows


In [5]:
from pyspark.sql import functions as F
als_candidates.groupBy('userId').agg(
    F.count(F.col('rating')).alias('num_rat')
).orderBy('num_rat').show(10)

+------+-------+
|userId|num_rat|
+------+-------+
| 83090|     58|
|118205|     69|
| 34576|     90|
|    65|    100|
|   458|    100|
|   879|    100|
|   883|    100|
|  1223|    100|
|  1977|    100|
|  2096|    100|
+------+-------+
only showing top 10 rows


The following users have < 100 ratings due to duplicate removals:
| User | Rating |
| 83090|         58|
|118205|         69|
| 34576|         90|

### Build User Features

In [6]:
from data.stats import compute_global_std

global_std = compute_global_std(train_sample)
global_std

1.0679534029605835

In [28]:
from features.user_features import build_user_features

user_feature = build_user_features(train_sample, global_std, k_shrinkage=20)
user_feature.show(5)

+------+------------------+-----------------+------------------+---------------------+------------------------+------------------+
|userId|   user_avg_rating|user_rating_count|   user_rating_std|user_log_rating_count|days_since_last_activity|    user_bayes_std|
+------+------------------+-----------------+------------------+---------------------+------------------------+------------------+
|   125| 3.669811320754717|               53|0.8317231662362131|   3.9889840465642745|                    1705|0.8964437790374105|
| 10641| 3.318725099601594|              251|1.0667745629360923|    5.529429087511423|                    4986|1.0668615621998925|
| 13042|3.7083333333333335|               24|0.9078961186825509|   3.2188758248682006|                    6501|0.9806494297180202|
| 30223|3.8050359712230217|              695| 0.695117154826979|     6.54534966033442|                    3019|0.7055461407887581|
| 45993|            4.1875|               16|0.9105858919765154|    2.8332133440562

## Item Features

In [8]:
from data.stats import compute_global_mean

mu = compute_global_mean(train_sample)
mu

3.523483257478244

In [29]:
# build out item features
from features.item_features import build_item_features

item_feature = build_item_features(train_sample, mu, C=50)
item_feature.show(5)

+-------+------------------+-----------------+------------------+---------------------+
|movieId|   item_avg_rating|item_rating_count| item_bayesian_avg|item_log_rating_count|
+-------+------------------+-----------------+------------------+---------------------+
|   7982|3.3333333333333335|                3|3.5127200542247587|   1.3862943611198906|
|  68135| 2.923076923076923|               13|3.3995898868874956|    2.639057329615259|
|  54190|               3.9|               10| 3.586236047898537|   2.3978952727983707|
|   2142| 2.638888888888889|               18| 3.289325924616356|   2.9444389791664403|
|   1580|3.5316666666666667|              300|3.5304976082111783|    5.707110264748875|
+-------+------------------+-----------------+------------------+---------------------+
only showing top 5 rows


## Biases

In [10]:
# Hyperparameters
bias_hparams = {
    'reg_param': 10,
    'tau': 3,
    'epsilon': 1e-6
}

### Item Biases

In [30]:
from features.biases import compute_item_bias

item_bias = compute_item_bias(item_feature, mu=mu, reg_param=bias_hparams['reg_param'])
item_bias.show(5)

+-------+--------------------+
|movieId|           item_bias|
+-------+--------------------+
|   7982|-0.04388075172574863|
|  68135| -0.3393601020529207|
|  54190| 0.18825837126087785|
|   2142| -0.5686678083788713|
|   1580|0.007919428246860514|
+-------+--------------------+
only showing top 5 rows


### User Biases

In [31]:
from features.biases import compute_user_bias

user_bias = compute_user_bias(train_sample, user_feature, item_bias, mu=mu, reg_param=bias_hparams['reg_param'])
user_bias.show(5)

+------+--------------------+
|userId|           user_bias|
+------+--------------------+
| 80033|-0.11742784003816066|
| 85321|-0.06847170700488094|
| 87656|0.011261681306202326|
| 35361| -0.7186960569484235|
| 34650|  0.3678423749105886|
+------+--------------------+
only showing top 5 rows


## Residuals & Weighting

### Expected Rating

In [32]:
from features.biases import compute_expected_rating

expected_rating = compute_expected_rating(train_sample, user_bias, item_bias, mu)
expected_rating.show(5)

+-------+------+------+--------------------+--------------------+------------------+
|movieId|userId|rating|           user_bias|           item_bias|   expected_rating|
+-------+------+------+--------------------+--------------------+------------------+
|     19| 80033|   3.5|-0.11742784003816066| -0.7608698941544753|2.6451855232856083|
|    344| 80033|   3.5|-0.11742784003816066| -0.5611960449445936|  2.84485937249549|
|   4447| 80033|   3.0|-0.11742784003816066|-0.29013344896415144| 3.115921968475932|
|     11| 80033|   3.5|-0.11742784003816066| 0.16948803460388065| 3.575543452043964|
|   7073| 80033|   4.0|-0.11742784003816066| 0.18400744112078035| 3.590062858560864|
+-------+------+------+--------------------+--------------------+------------------+
only showing top 5 rows


## Weightings

In [33]:
from features.biases import compute_user_weights

weights = compute_user_weights(
    expected_rating, user_feature, tau=bias_hparams['tau'],epsilon=bias_hparams['epsilon'])
weights.show(5)

+------+-------+--------------------+
|userId|movieId|              weight|
+------+-------+--------------------+
| 80033|     19| 0.29033924162165753|
| 80033|    344| 0.22518257352388543|
| 80033|   4447|-0.04051681426527...|
| 80033|     11|-0.02641211431353...|
| 80033|   7073| 0.14238480349659136|
+------+-------+--------------------+
only showing top 5 rows


## Tag Features

### PCA dimension reduction

In [34]:
import configs.settings as cfg
from features.tag_features import build_genome_pca_features
# load csv as dataframe
genomes = spark.read.csv("../data/raw/genome_scores.csv", schema=cfg.GENOMIC_SCHEMA, header=True)

# conduct pca and scaling on genome features
genome_scalar_model, genome_pca_df, genome_pca_model =  (
    build_genome_pca_features(genomes, k=45)
)

In [6]:
# Saving genome pca dataframe
genome_pca_df.write.mode('overwrite').parquet('../data/features/genome_pca_45.parquet')

### Normalise Tags

In [35]:
from data.preprocessing import normaliser
item_tag_norm = normaliser(genome_pca_df, input_col='pca_features', output_col='item_tag_norm')
item_tag_norm.show(5)

+-------+--------------------+
|movieId|       item_tag_norm|
+-------+--------------------+
|    496|[0.07888750859851...|
|    148|[-0.0275135375140...|
|    463|[-0.6914273308849...|
|    471|[0.40777863452039...|
|    833|[-0.6868075439068...|
+-------+--------------------+
only showing top 5 rows


## User Vector

In [36]:
genome_pca_df.cache()
genome_pca_df.show(5)

+-------+--------------------+
|movieId|        pca_features|
+-------+--------------------+
|    496|[0.20513351766232...|
|    148|[-0.0423306077345...|
|    463|[-1.6947855038298...|
|    471|[1.49517609789605...|
|    833|[-2.2212663724634...|
+-------+--------------------+
only showing top 5 rows


In [37]:
from features.user_vectors import build_user_tags

user_tags = build_user_tags(weights, genome_pca_df)
user_tags.show(5)

+------+--------------------+
|userId|            user_tag|
+------+--------------------+
|    49|[0.27824682880819...|
|   101|[0.50514107959677...|
|   125|[0.26997927680381...|
|   166|[1.02261154442334...|
|   296|[-0.4821795996254...|
+------+--------------------+
only showing top 5 rows


In [38]:
from data.preprocessing import normaliser

user_tag_norm = normaliser(user_tags, input_col='user_tag', output_col='user_tag_norm')
user_tag_norm.show(5)

+------+--------------------+
|userId|       user_tag_norm|
+------+--------------------+
|    49|[0.30509359649352...|
|   101|[0.75200454646163...|
|   125|[0.42958727147964...|
|   166|[0.58777397923818...|
|   296|[-0.4278544692772...|
+------+--------------------+
only showing top 5 rows


## Cosine Similarity
Compare user average tag (genome scores) to films

In [39]:
from ranking.tag_similarity import compute_tag_similarity

similarity_score = compute_tag_similarity(user_tag_norm, item_tag_norm, als_candidates)

similarity_score.orderBy('userId').show(150)

+------+-------+----------+--------------------+
|userId|movieId| als_score|          similarity|
+------+-------+----------+--------------------+
|    49|    318|0.88383806| 0.08045937103692147|
|    49|    858|0.73075855|  0.3836213092510559|
|    49|   2959|0.72190726| 0.46885730401225256|
|    49|    593|  0.714609| 0.28127126757859156|
|    49|  79132|0.67034775|  0.3229787179337787|
|    49|   2329| 0.6610851|  0.2597038265939558|
|    49|   2571|0.65427977|  0.2224152101848002|
|    49|   1221| 0.6513068|  0.2909159166655768|
|    49|   6016| 0.6484524| 0.35456451882223816|
|    49|    431| 0.6026051|  0.5084674909637065|
|    49|   1193| 0.6001093| 0.12341001811684552|
|    49|   2858|0.59119076|  0.2566540574986964|
|    49|   4226| 0.5746324|  0.5030951528376131|
|    49|   1405|0.57437676|0.057763245013237166|
|    49|   1213|0.55044585|  0.3909781624280554|
|    49|    543|0.51556045|-0.15084159902705083|
|    49|    110|0.51379126|-0.09000260518374774|
|    49|    356| 0.5

# Final Feature Dataset

In [40]:
from ranking.ranking_df import create_ranking_df

ranking = create_ranking_df(similarity_score, user_feature, item_feature, user_bias, item_bias)
ranking.orderBy('userId').show(200)

+-------+------+----------+--------------------+---------------+------------------+---------------------+------------------------+------------------+------------------+------------------+---------------------+-------------------+--------------------+
|movieId|userId| als_score|          similarity|user_avg_rating|   user_rating_std|user_log_rating_count|days_since_last_activity|    user_bayes_std|   item_avg_rating| item_bayesian_avg|item_log_rating_count|          user_bias|           item_bias|
+-------+------+----------+--------------------+---------------+------------------+---------------------+------------------------+------------------+------------------+------------------+---------------------+-------------------+--------------------+
|    318|    49|0.88383806| 0.08045937103692147|           3.75|0.7071067811865476|   3.2188758248682006|                     691|0.8711279729020184| 4.447368421052632| 4.367996843425966|    6.278521424165844|0.04272641608774827|   0.9068393118479